In [42]:
# setup from 01_tdbrain_eda.ipynb

from pathlib import Path
from collections import defaultdict
import pandas as pd

data_root = Path.cwd().parent / "data"
participants_file = data_root / "TDBRAIN_participants_V3.xlsx"
part_df = pd.read_excel(participants_file)

neo_cols = [f'neoFFI_q{i}' for i in range(1, 61)]
filt_df = part_df[
    (part_df["formal_status"] != "UNKNOWN") &
    (part_df["age"].notna()) &
    (part_df["gender"].notna()) &
    (part_df["education"].notna()) &
    (part_df["n_oddb_CP"].notna()) &
    (part_df["n_oddb_FP"].notna()) &
    (part_df["n_oddb_CN"].notna()) &
    (part_df["n_oddb_FN"].notna()) &
    (part_df["avg_rt_oddb_CP"].notna()) &
    (part_df[neo_cols].notna().all(axis=1))
]

filt_ids = set(filt_df["TDBRAIN_ID"])
bdf_index = defaultdict(list)
for path in Path(data_root).rglob("*.bdf"):
    subj_id = path.name.split("_")[0]
    if subj_id in filt_ids and "ses-1" in path.parts:
        bdf_index[subj_id].append(path)

In [18]:
import mne
import json
mne.set_log_level('WARNING')

In [51]:
def process_bdf(bdf_path):
    """Processes a single bdf path from end to end, outputting the structured text that the LLM will see."""
    # stores the raw data in raw_bdf
    raw_bdf = mne.io.read_raw_bdf(bdf_path, preload=True)
    # creates a copy (to preserve the original raw signal) and only selects the EEG channels
    bdf = raw_bdf.copy().pick('eeg')

    non_eeg = ['VPVA', 'VNVB', 'HPHL', 'HNHR', 'Erbs', 'Mass']
    # some non eeg channels still remain after picking eeg due to a metadata mismatch, so this removes those as well
    bdf.drop_channels([ch for ch in non_eeg if ch in bdf.ch_names])
    # applies a bandpass filter (1-40 Hz) in order to remove movement artifacts and other noise
    bdf.filter(l_freq=1.0, h_freq=40.0)
    # transforms into signal power distributed across specific frequencies 
    spectrum = bdf.compute_psd(method='welch', fmin=1, fmax=40.0)
    # extract the psd data and the frequency ranges
    psds, freqs = spectrum.get_data(return_freqs=True)
    band_powers = {}
    # the frequency ranges and their corresponding band names
    freq_ranges = {"delta": (0.5, 4.0), "theta": (4.0, 8.0), "alpha": (8.0, 13.0), "beta": (13.0, 30.0), "gamma": (30.0, 100.0)}
    for band, (fmin, fmax) in freq_ranges.items():
        # create a boolean mask for the freq range in question
        mask = (freqs >= fmin) & (freqs <= fmax)
        # in the psds data, keeps all dimensions same (through the ...), apply mask to the last dim (keeping only the freqs within the mask)
        # and then taskes the mean across that very last dimension (using -1)
        # finally, the multiplication by 1e12 converts these into microvolts squared to increase LLM comprehension with the numbers
        band_powers[band] = psds[..., mask].mean(axis=-1) * 1e12
    # returning not only the band powers but also the names of the channels
    return band_powers, bdf.ch_names
process_bdf(bdf_index['sub-88045085'][0])[1]

['Fp1',
 'Fp2',
 'F7',
 'F3',
 'Fz',
 'F4',
 'F8',
 'FC3',
 'FCz',
 'FC4',
 'T7',
 'C3',
 'Cz',
 'C4',
 'T8',
 'CP3',
 'CPz',
 'CP4',
 'P7',
 'P3',
 'Pz',
 'P4',
 'P8',
 'O1',
 'Oz',
 'O2']

In [59]:
neural_data = {}
for subj_index, (subj_id, bdf_paths) in enumerate(bdf_index.items()):
    record = ""
    for bdf_path in bdf_paths:
        band_powers, channel_names = process_bdf(bdf_path)
        if "restEC" in bdf_path.name:
            record += "Eyes Closed Spectral Features (μV²/Hz):\n"
        elif "restEO" in bdf_path.name:
            record += "Eyes Open Spectral Features (μV²/Hz):\n"
        else:
            print(subj_id)
            print(bdf_paths)
            continue
        for index, channel in enumerate(channel_names):
                record += "Channel " + channel + ": "
                for band, powers in band_powers.items():
                    record += band + "=" + str(round(powers[index], 2)) + ", "
    neural_data[subj_id] = record
    if (subj_index % 10) == 0:
         print(subj_index, "subjects processed out of", len(bdf_index))

with open(data_root / "records.json", "w", encoding="utf-8") as file:
    json.dump(neural_data, file, indent=2)

0 subjects processed out of 355
10 subjects processed out of 355
20 subjects processed out of 355
30 subjects processed out of 355
40 subjects processed out of 355
50 subjects processed out of 355
60 subjects processed out of 355
70 subjects processed out of 355
80 subjects processed out of 355
90 subjects processed out of 355
100 subjects processed out of 355
110 subjects processed out of 355
120 subjects processed out of 355
130 subjects processed out of 355
140 subjects processed out of 355
150 subjects processed out of 355
160 subjects processed out of 355
170 subjects processed out of 355
180 subjects processed out of 355
190 subjects processed out of 355
200 subjects processed out of 355
210 subjects processed out of 355
220 subjects processed out of 355
230 subjects processed out of 355
240 subjects processed out of 355
250 subjects processed out of 355
260 subjects processed out of 355
270 subjects processed out of 355
280 subjects processed out of 355
290 subjects processed ou

In [43]:
print(len(bdf_index))
print(len(set(bdf_index.keys())))
print(filt_df["TDBRAIN_ID"].nunique())

355
355
360


In [44]:
print(len(filt_df))
print(filt_df["TDBRAIN_ID"].nunique())

375
360
